3.21更新：
这个是计算互信息的函数。根据sklearn函数的引用，我们其实只需要处理y为一维的情况，且获得的是结果中第一个值。
在保证输入和输出不变的前提下，使用torch进行下列函数的计算以替代功能。

In [1]:
import torch

def mutual_info_regression(x, y, n_neighbors=3):
    """
    Using PyTorch to calculate the mutual information between continuous target variable y and feature x without relying on sklearn at all.
    """
    def knn_distances(x, k):
        """
        Calculate the K nearest neighbor distance between given data points X.
        """
        n = x.size(0)
        # 计算所有点之间的距离
        xx = x.pow(2).sum(1, keepdim=True).expand(n, n)
        dist = xx + xx.t() - 2 * x.mm(x.t())
        # 保证对角线为无穷大，避免自己成为最近邻
        dist = dist + torch.eye(n).to(x.device) * 1e10
        # 取每一行的前k个最小值，即k个最近邻的距离
        distances, _ = dist.topk(k=k, largest=False, dim=1)
        return distances[:, -1]  # 返回第k个最近邻的距离

    # 确保x和y是2D和1D张量
    if len(x.shape) == 1:
        x = x.reshape(-1, 1)
    if len(y.shape) > 1:
        y = y.flatten()

    # 合并x和y以计算它们的联合分布
    xy = torch.cat((x, y.reshape(-1, 1)), dim=1)

    # 计算距离
    rho = 0.4  # 常数，用于距离转换
    dx = knn_distances(x, n_neighbors) * rho
    dy = knn_distances(y.reshape(-1, 1), n_neighbors) * rho
    dxy = knn_distances(xy, n_neighbors) * rho

    # 计算互信息
    mi = torch.digamma(torch.tensor(n_neighbors)) + torch.digamma(torch.tensor(len(x))) - (torch.mean(torch.digamma(dx+1)) + torch.mean(torch.digamma(dy+1)) - torch.mean(torch.digamma(dxy+1)))

    return mi.item()  # 返回Python标量

3.23更新：
test一下上面代码和原引用的输出差距。

In [2]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.feature_selection import mutual_info_regression

# 生成一些合成数据
X, y = make_regression(n_samples=100, n_features=1, noise=0.1, random_state=42)

# 使用sklearn计算互信息
mi_sklearn = mutual_info_regression(X, y)
print("sklearn:", mi_sklearn[0])

# 使用PyTorch实现计算互信息
mi_torch = mutual_info_regression(X, y, n_neighbors=3)
print("PyTorch:", mi_torch)

# 比较两种方法的输出结果
print("difference:", np.abs(mi_sklearn[0] - mi_torch))

sklearn: 3.3148775176396215
PyTorch: [3.31487752]
difference: [0.]


再引用V_bcp的方法试一下。

In [3]:
import torchvinecopulib as tvc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
fam, par, rot = "Clayton", (1,), 90
V_bcp = tvc.bicop.DataBiCop(fam=fam, par=par, rot=rot).sim(
    num_sim=20000, device=DEVICE, seed=0
)

V_bcp

tensor([[0.9701, 0.1370],
        [0.4594, 0.9277],
        [0.6450, 0.7407],
        ...,
        [0.7533, 0.7425],
        [0.5787, 0.6669],
        [0.6113, 0.5218]], dtype=torch.float64)

In [4]:
# 将V_bcp张量转换为NumPy数组，用于sklearn的函数
V_bcp_numpy = V_bcp.numpy()

X_sklearn = V_bcp_numpy[:, 0].reshape(-1, 1)  # 选择第一列作为特征
y_sklearn = V_bcp_numpy[:, 1]  # 选择第二列作为目标
mi_sklearn = mutual_info_regression(X_sklearn, y_sklearn)

X_torch = V_bcp[:, 0].unsqueeze(1)  # 选择第一列作为特征并确保为2D张量
y_torch = V_bcp[:, 1]  # 选择第二列作为目标
mi_torch = mutual_info_regression(X_torch, y_torch, n_neighbors=3)

print("sklearn:", mi_sklearn[0])
print("PyTorch:", mi_torch)
print("difference:", np.abs(mi_sklearn[0] - mi_torch))

sklearn: 0.20340393516834254
PyTorch: [0.20340394]
difference: [0.]
